# Пустая серверная лента — 26 октября 2026

Пользователь сообщил о пустой ленте у Сергея (РЖД). Проверка штатным HTTPS API показала пустые ленты у всех четырёх основных тестовых пользователей. Сессии получены для чтения данных при диагностике; отдельные проверки паролей восьми пользователей не выполнялись.

У каждой компании все четыре документа имеют queued / model_not_configured, attempts=0; news_processed_through=null. Это подтверждённый блокер персонального анализа: UnconfiguredModel в orchestrator/analytics/runtime.py устанавливается, когда LLMClient не получает действующий AI_API_KEY. Загруженный общий корпус не равен готовым персональным карточкам.

В Helm у analytics и analytics-worker уже есть envFrom secretRef alpha-secrets. Прежнее утверждение об отсутствии передачи ключей было ошибочным: секрет импортируется целиком. Jev использует отдельный AI_SYSTEMONE_API_KEY и не заменяет ключ аналитической LLM.

Подготовлен ignored .env.k8s-secret-patch.json с AI_API_KEY из локально сохранённой конфигурации, права 0600. Ключ не включён в notebook, Git или вывод. Файл не применён: kubectl config get-contexts вернул пустой список, SSH-подключение не предоставлено. Для исправления требуется patch alpha-secrets и restart alpha-analytics / alpha-analytics-worker; затем проверка возобновления обработки. Это первый доказанный блокер, не подтверждение отсутствия других ошибок.


In [1]:
import json
observations = [{'persona': 'x5', 'company_id': 8, 'user_id': 9, 'feed_http': 200, 'feed_count': 0, 'documents_http': 200, 'documents': [{'filename': '01_financial_plan.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '02_liquidity_and_debt.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '03_treasury_questions.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '04_commercial_demand.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}], 'news_processed_through': None}, {'persona': 'rzd', 'company_id': 10, 'user_id': 11, 'feed_http': 200, 'feed_count': 0, 'documents_http': 200, 'documents': [{'filename': '01_critical_items.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '02_supply_routes.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '03_procurement_playbook.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '04_repair_schedule.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}], 'news_processed_through': None}, {'persona': 'beeline', 'company_id': 9, 'user_id': 10, 'feed_http': 200, 'feed_count': 0, 'documents_http': 200, 'documents': [{'filename': '01_strategy_brief.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '02_pilot_economics.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '03_decision_questions.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '04_product_readiness.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}], 'news_processed_through': None}, {'persona': 'etalon', 'company_id': 13, 'user_id': 13, 'feed_http': 200, 'feed_count': 0, 'documents_http': 200, 'documents': [{'filename': '01_sales_segments.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '02_demand_signals.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '03_commercial_questions.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '04_financial_sensitivity.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}], 'news_processed_through': None}]
print(json.dumps(observations,ensure_ascii=False,indent=2))


[
  {
    "persona": "x5",
    "company_id": 8,
    "user_id": 9,
    "feed_http": 200,
    "feed_count": 0,
    "documents_http": 200,
    "documents": [
      {
        "filename": "01_financial_plan.txt",
        "status": "queued",
        "error": "model_not_configured",
        "attempts": 0
      },
      {
        "filename": "02_liquidity_and_debt.txt",
        "status": "queued",
        "error": "model_not_configured",
        "attempts": 0
      },
      {
        "filename": "03_treasury_questions.txt",
        "status": "queued",
        "error": "model_not_configured",
        "attempts": 0
      },
      {
        "filename": "04_commercial_demand.txt",
        "status": "queued",
        "error": "model_not_configured",
        "attempts": 0
      }
    ],
    "news_processed_through": null
  },
  {
    "persona": "rzd",
    "company_id": 10,
    "user_id": 11,
    "feed_http": 200,
    "feed_count": 0,
    "documents_http": 200,
    "documents": [
      {
        "fil

## Повторная проверка после сообщения о добавлении ключа

Pull main выполнен: Already up to date. У Сергея РЖД GET feed снова вернул HTTP 200 / [], issues — []. Все четыре integration_file по-прежнему queued / model_not_configured / attempts=0. news_processed_through=null. Ошибка сохранена сервером; текущие переменные pod без доступа к кластеру не проверены.

Гипотеза о сутках проверена статически: frontend/report.ts dailySummary использует 24 часа только для сводки; FeedPage и feedFilters не фильтруют ленту по дате. Store.Feed выбирает active записи пользователя/роли без временного условия; окно Go обработки в config/Helm — 504h. Возможное операторское переопределение окружения без kubectl не проверено.


## Уточнение: воркер начал обращения к LLM

Повторный опрос в 16:17 МСК показал у первых двух документов X5 queued / connection / attempts=1. Таким образом, прежняя ошибка model_not_configured у РЖД является сохранённым состоянием и не доказывает отсутствия ключа в новом процессе. Текущий наблюдаемый блокер у обрабатываемых документов — соединение с LLM. Worker синхронно обходит документы компаний, поэтому ошибки остальных записей обновляются позже.

Settings по умолчанию использует https://changellenge.up.railway.app/v1. В инструкции организаторов также есть резервный адрес https://api.alfahack-gateway26.ru/v1 для российских хостингов; текущий валидатор Settings разрешает только основной адрес. Это кандидат для проверки, не доказанная причина connection и не выполненное переключение. Для установления сетевой причины запрошен GET /models из analytics-worker; доступ к kubeconfig/SSH в текущей сессии отсутствует. Изменение окна 21 день не выполнялось.


In [2]:
import json
after_restart = [{'persona': 'x5', 'observed_at': '2026-10-26T13:17:09.410573+00:00', 'documents': [{'filename': '01_financial_plan.txt', 'status': 'queued', 'error': 'connection', 'attempts': 1}, {'filename': '02_liquidity_and_debt.txt', 'status': 'queued', 'error': 'connection', 'attempts': 1}, {'filename': '03_treasury_questions.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '04_commercial_demand.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}]}, {'persona': 'beeline', 'observed_at': '2026-10-26T13:17:09.730440+00:00', 'documents': [{'filename': '01_strategy_brief.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '02_pilot_economics.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '03_decision_questions.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '04_product_readiness.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}]}, {'persona': 'rzd', 'observed_at': '2026-10-26T13:17:10.052475+00:00', 'documents': [{'filename': '01_critical_items.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '02_supply_routes.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '03_procurement_playbook.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '04_repair_schedule.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}]}, {'persona': 'etalon', 'observed_at': '2026-10-26T13:17:10.364592+00:00', 'documents': [{'filename': '01_sales_segments.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '02_demand_signals.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '03_commercial_questions.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}, {'filename': '04_financial_sensitivity.txt', 'status': 'queued', 'error': 'model_not_configured', 'attempts': 0}]}]
print(json.dumps(after_restart,ensure_ascii=False,indent=2))


[
  {
    "persona": "x5",
    "observed_at": "2026-10-26T13:17:09.410573+00:00",
    "documents": [
      {
        "filename": "01_financial_plan.txt",
        "status": "queued",
        "error": "connection",
        "attempts": 1
      },
      {
        "filename": "02_liquidity_and_debt.txt",
        "status": "queued",
        "error": "connection",
        "attempts": 1
      },
      {
        "filename": "03_treasury_questions.txt",
        "status": "queued",
        "error": "model_not_configured",
        "attempts": 0
      },
      {
        "filename": "04_commercial_demand.txt",
        "status": "queued",
        "error": "model_not_configured",
        "attempts": 0
      }
    ]
  },
  {
    "persona": "beeline",
    "observed_at": "2026-10-26T13:17:09.730440+00:00",
    "documents": [
      {
        "filename": "01_strategy_brief.txt",
        "status": "queued",
        "error": "model_not_configured",
        "attempts": 0
      },
      {
        "filename": "

## Подтверждённая причина заголовка Authorization

Пользователь предоставил traceback из analytics-worker: httpcore/httpx LocalProtocolError: Illegal header value, значение Authorization заканчивается переводом строки. Сам ключ и сырой traceback в notebook не сохранены. В данном случае connection возникла при формировании HTTP-запроса, а не доказывала сетевую недоступность Railway. Смена шлюза по этим данным не обоснована.

require_key проверял api_key.get_secret_value().strip(), но LLMClient передавал SDK исходное значение. В Settings добавлена нормализация api_key через field_validator: удаляются внешние пробельные символы до вычисления account hash и передачи SDK. Регрессионный тест с синтетическим ключом для str/SecretStr воспроизвёл ошибку (RED), после исправления оба подслучая прошли.

Проверки основной моделью: 14 тестов test_llm — OK; полный orchestrator/tests — 129 tests, OK, skipped=9 (PostgreSQL DSN не задан). git diff --check — OK. Реальные платные вызовы для проверки исправления не выполнялись. Патч Kubernetes Secret проверен на синтетических данных; он меняет только AI_API_KEY. Код исправлен локально, коммит/пуш/деплой не выполнялись. Ключ в живом Secret требует удаления конечного перевода строки и перезапуска двух Python deployment.
